# YOLOv5 Custom Training for Robot Course Detection

Train a YOLOv5 model to detect course elements:
- **line** — black line on white ground (straight + curves)
- **red_sign** — red rectangle (traffic sign)
- **green_sign** — green rectangle (traffic sign)
- **cross** — cross symbol
- **circle** — circle symbol

Pipeline: Annotate → Train YOLOv5 → Export ONNX → Compile to Hailo `.hef`

## 1. Setup

In [1]:
!pip install ultralytics label-studio-sdk onnx onnxsim

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA/ROCm available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

PyTorch: 2.12.0.dev20260307+rocm7.2
CUDA/ROCm available: True
Device: Radeon RX 7900 XTX


## 2. Dataset Structure

Organize your captured images into this structure:
```
dataset/
├── images/
│   ├── train/     # ~80% of images
│   └── val/       # ~20% of images
├── labels/
│   ├── train/     # YOLO .txt annotations (same filenames)
│   └── val/
└── dataset.yaml
```

### Annotation
Use [Label Studio](https://labelstud.io/) or [CVAT](https://www.cvat.ai/) to annotate bounding boxes.  
Export in **YOLO format** — one `.txt` per image with lines:  
`class_id cx cy w h` (normalized 0-1)

Run `capture_data.py` on the Pi to collect images from the course.

In [2]:
import os
from pathlib import Path

# Adjust this to where your dataset lives
DATASET_ROOT = Path("dataset")

# Create directory structure
for split in ["train", "val"]:
    (DATASET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Dataset directories created.")
print(f"Put training images in:   {DATASET_ROOT / 'images' / 'train'}")
print(f"Put validation images in: {DATASET_ROOT / 'images' / 'val'}")
print(f"Put YOLO label .txt files in the matching labels/ directories.")

Dataset directories created.
Put training images in:   dataset/images/train
Put validation images in: dataset/images/val
Put YOLO label .txt files in the matching labels/ directories.


In [3]:
# Create dataset.yaml
dataset_yaml = DATASET_ROOT / "dataset.yaml"
dataset_yaml.write_text(f"""path: {DATASET_ROOT.resolve()}
train: images/train
val: images/val

names:
  0: line
  1: red_sign
  2: green_sign
  3: cross
  4: circle
""")
print(f"Written: {dataset_yaml}")
print(dataset_yaml.read_text())

Written: dataset/dataset.yaml
path: /run/host/home/timon/Develop/rpi-nxt2/src/host/python/apps/hailo_mv/training/dataset
train: images/train
val: images/val

names:
  0: line
  1: red_sign
  2: green_sign
  3: cross
  4: circle



## 3. Train YOLOv5

Using `ultralytics` (YOLOv8 API which also supports YOLOv5-style models).  
We use `yolov5n` (nano) since it needs to run on the Hailo-8 with low latency.

In [ ]:
from ultralytics import YOLO

# Load YOLOv5n pretrained on COCO as starting point
model = YOLO("yolov5n.pt")

# Train on your custom dataset
results = model.train(
    data=str(dataset_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,           # GPU
    patience=20,         # early stopping
    name="course_det",
)

In [ ]:
# Validate the trained model
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

## 4. Export to ONNX

The Hailo Dataflow Compiler (DFC) takes ONNX as input.

In [ ]:
# Export best weights to ONNX
best_weights = Path("runs/detect/course_det/weights/best.pt")
model = YOLO(str(best_weights))
model.export(format="onnx", imgsz=640, simplify=True, opset=12)

onnx_path = best_weights.with_suffix(".onnx")
print(f"ONNX model saved to: {onnx_path}")
print(f"Size: {onnx_path.stat().st_size / 1024 / 1024:.1f} MB")

## 5. Compile to Hailo `.hef`

This step requires the **Hailo Dataflow Compiler (DFC)**, which runs on x86 Linux.  
Install it from [Hailo Developer Zone](https://hailo.ai/developer-zone/).

```bash
pip install hailo_dataflow_compiler
```

### Option A: Using Hailo Model Zoo CLI
```bash
# Parse ONNX → Hailo Archive (.har)
hailo parser onnx best.onnx --net-name course_det

# Optimize (quantize to int8) — needs calibration images
hailo optimize course_det.har --calib-path dataset/images/train

# Compile to .hef for Hailo-8
hailo compiler course_det_quantized.har --hw-arch hailo8
```

### Option B: Python API

In [ ]:
# Uncomment and run if hailo_dataflow_compiler is installed
# from hailo_sdk_client import ClientRunner
#
# onnx_path = "runs/detect/course_det/weights/best.onnx"
# runner = ClientRunner(hw_arch="hailo8")
#
# # Parse ONNX
# hn, npz = runner.translate_onnx_model(
#     onnx_path,
#     net_name="course_det",
#     start_node_names=["images"],
#     end_node_names=["output0"],
# )
# runner.save_har("course_det.har")
#
# # Optimize (quantize) — provide calibration images
# calib_dataset = runner.load_calibration_dataset(
#     "dataset/images/train", data_count=64
# )
# runner.optimize(calib_dataset)
# runner.save_har("course_det_quantized.har")
#
# # Compile
# hef = runner.compile()
# with open("course_det.hef", "wb") as f:
#     f.write(hef)
# print("course_det.hef written!")

## 6. Deploy on Raspberry Pi

Copy `course_det.hef` to the Pi and update the model path:

```python
# hailo_mv.py
model_path = "/path/to/course_det.hef"
```

The `RobotVision` class will then run your custom model on the Hailo-8 chip.  
You'll still need post-processing (NMS + box drawing) in `robot_vision.py`,  
but now the detections will be for your 5 course classes.